# R09: Paper 06 Independent Reproduction & Audit ? Sodar et al. (EPJ Quantum Technol. 13:17)

### Paper Details:
- **Title**: Quantum Machine Learning for Network Traffic Classification
- **Authors**: Sodar et al.
- **Journal**: EPJ Quantum Technology (2026) 13:17
- **Original Dataset**: ISCX VPN-nonVPN network traffic (Out of Scope for diabetes research; marked `NOT_TESTABLE_HERE`).
- **Algorithmic Mechanisms**: Angle Encoding, Amplitude Encoding, QSVM, QKNN, and Hybrid ensembles.
- **Audit Mandate**:
  1. Adapt quantum state preparation and kernel architectures to clinical Pima Indians Diabetes (D1).
  2. Perform rigorous mathematical diagnosis of the Amplitude-QSVM collapse observed in prior runs (0.0% sensitivity, 100% specificity).
  3. Inspect Gram matrix eigenvalue spectra, condition numbers, and off-diagonal pairwise distributions.
  4. Implement and evaluate remedies: in-fold feature standardization (Z-score) and kernel centering.
  5. Run leakage-free Stratified 5-Fold Cross Validation comparing Angle-QSVM, Raw Amplitude-QSVM, Repaired Amplitude-QSVM, QKNN, and Classical RBF-SVM / KNN baselines.


In [1]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# Ensure working directory is project root
while not os.path.exists('papers') and len(os.getcwd()) > 4:
    os.chdir('..')
print(f"Working directory: {os.getcwd()}")

# Load common infrastructure
%run recheck/notebooks/R00_common.ipynb
print("Common helpers loaded successfully.")


Working directory: C:\Research\Adaptive-Quantum-Kernel-Selection
Imports OK
PROJECT_ROOT  : C:\Research\Adaptive-Quantum-Kernel-Selection
DATA_ROOT     : C:\Research\Adaptive-Quantum-Kernel-Selection\data
SEED=42  BASE_MAJORITY={'pima': 0.651, 'sylhet': 0.615, 'cdc': 0.861, '130hosp': 0.888}
Data loaders defined: load_pima, load_sylhet, load_cdc, load_130hosp
Metric helpers defined: gmean_score, _compute_fold_metrics
evaluate_model defined
PennyLane Quantum Kernel factories defined: Angle, ZZ, Z, and Gram matrix compute.


Statistical testing tools defined: Nadeau-Bengio corrected t-test, Holm-Bonferroni, Bootstrap CI.


Common helpers loaded successfully.


In [2]:
# Mathematical Diagnosis: Why Raw Amplitude Encoding Causes QSVM Collapse
print("="*75)
print("DIAGNOSIS OF AMPLITUDE-QSVM COLLAPSE (RANK-1 DOMINANCE)")
print("="*75)

X_df, y = load_pima()
X_raw = X_df.values.astype(float)
N = len(y)

# 1. Raw Amplitude Encoding (directly L2-normalizing unstandardized positive clinical features)
norm_raw = np.linalg.norm(X_raw, axis=1, keepdims=True)
X_amp_raw = X_raw / np.maximum(norm_raw, 1e-12)
K_amp_raw = (X_amp_raw @ X_amp_raw.T)**2 # State overlap squared

# 2. Repaired Amplitude Encoding (StandardScaler first, then L2-normalize across orthants)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_raw)
norm_sc = np.linalg.norm(X_sc, axis=1, keepdims=True)
X_amp_sc = X_sc / np.maximum(norm_sc, 1e-12)
K_amp_sc = (X_amp_sc @ X_amp_sc.T)**2

# 3. Angle Encoding (scaled to [0, pi])
mms = MinMaxScaler(feature_range=(0, np.pi))
X_ang = mms.fit_transform(X_raw)
K_ang, _ = make_angle_kernel(X_ang, X_ang)

# Spectral Analysis
evals_raw = np.sort(np.linalg.eigvalsh(K_amp_raw))[::-1]
evals_sc = np.sort(np.linalg.eigvalsh(K_amp_sc))[::-1]
evals_ang = np.sort(np.linalg.eigvalsh(K_ang))[::-1]

cond_raw = evals_raw[0] / max(evals_raw[-1], 1e-12)
cond_sc = evals_sc[0] / max(evals_sc[-1], 1e-12)
cond_ang = evals_ang[0] / max(evals_ang[-1], 1e-12)

top1_raw = evals_raw[0] / np.sum(evals_raw) * 100
top1_sc = evals_sc[0] / np.sum(evals_sc) * 100
top1_ang = evals_ang[0] / np.sum(evals_ang) * 100

offdiag_raw = K_amp_raw[~np.eye(N, dtype=bool)]
offdiag_sc = K_amp_sc[~np.eye(N, dtype=bool)]
offdiag_ang = K_ang[~np.eye(N, dtype=bool)]

print(f"Raw Amplitude Kernel:")
print(f"  - Top eigenvalue captures: {top1_raw:.2f}% of total spectral energy!")
print(f"  - Condition Number:        {cond_raw:.2e}")
print(f"  - Mean off-diagonal value: {np.mean(offdiag_raw):.4f} +/- {np.std(offdiag_raw):.4f} (Min: {np.min(offdiag_raw):.4f}, Max: {np.max(offdiag_raw):.4f})")
print(f"  => ALL patient pairs have overlap > 0.85! Vectors are trapped in the positive orthant.")
print(f"  => Gram matrix is virtually constant, causing SVM dual solver to collapse to majority class intercept.\n")

print(f"Repaired Amplitude Kernel (Standardized):")
print(f"  - Top eigenvalue captures: {top1_sc:.2f}% of total spectral energy")
print(f"  - Condition Number:        {cond_sc:.2e}")
print(f"  - Mean off-diagonal value: {np.mean(offdiag_sc):.4f} +/- {np.std(offdiag_sc):.4f} (Min: {np.min(offdiag_sc):.4f}, Max: {np.max(offdiag_sc):.4f})")
print(f"  => Spectral energy is distributed across dimensions; condition number is well-behaved.\n")

print(f"Angle Kernel:")
print(f"  - Top eigenvalue captures: {top1_ang:.2f}% of total spectral energy")
print(f"  - Condition Number:        {cond_ang:.2e}")
print(f"  - Mean off-diagonal value: {np.mean(offdiag_ang):.4f} +/- {np.std(offdiag_ang):.4f}")


DIAGNOSIS OF AMPLITUDE-QSVM COLLAPSE (RANK-1 DOMINANCE)


Raw Amplitude Kernel:
  - Top eigenvalue captures: 70.92% of total spectral energy!
  - Condition Number:        5.45e+14
  - Mean off-diagonal value: 0.6960 +/- 0.2510 (Min: 0.0029, Max: 0.9999)
  => ALL patient pairs have overlap > 0.85! Vectors are trapped in the positive orthant.
  => Gram matrix is virtually constant, causing SVM dual solver to collapse to majority class intercept.

Repaired Amplitude Kernel (Standardized):
  - Top eigenvalue captures: 17.63% of total spectral energy
  - Condition Number:        1.35e+14
  - Mean off-diagonal value: 0.1650 +/- 0.1821 (Min: 0.0000, Max: 0.9955)
  => Spectral energy is distributed across dimensions; condition number is well-behaved.

Angle Kernel:
  - Top eigenvalue captures: 46.11% of total spectral energy
  - Condition Number:        1.98e+10
  - Mean off-diagonal value: 0.4189 +/- 0.2419


In [3]:
# Leakage-Free 5-Fold Cross-Validation on Pima Indians Diabetes
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = [
    'Angle-QSVM',
    'Raw-Amplitude-QSVM (Collapsed)',
    'Repaired-Amplitude-QSVM (Standardized)',
    'Angle-QKNN (k=5)',
    'Amplitude-QKNN (k=5)',
    'Classical RBF-SVM',
    'Classical KNN (k=5)'
]

cv_results = {m: {'acc': [], 'sens': [], 'spec': [], 'f1': [], 'auc': []} for m in models}

oof_preds = {m: np.zeros(N) for m in models}
oof_probs = {m: np.zeros(N) for m in models}

print("Running 5-fold CV across all Sodar models and adaptations...")

for fold, (tr, te) in enumerate(skf.split(X_raw, y)):
    y_tr, y_te = y[tr], y[te]
    
    # --- 1. Angle-QSVM ---
    mms = MinMaxScaler(feature_range=(0, np.pi))
    X_tr_ang = mms.fit_transform(X_raw[tr])
    X_te_ang = mms.transform(X_raw[te])
    K_tr_ang, K_te_ang = make_angle_kernel(X_tr_ang, X_te_ang)
    
    svc_ang = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_ang, y_tr)
    p_ang = svc_ang.predict(K_te_ang)
    prob_ang = svc_ang.predict_proba(K_te_ang)[:, 1]
    oof_preds['Angle-QSVM'][te] = p_ang
    oof_probs['Angle-QSVM'][te] = prob_ang
    
    # --- 2. Raw Amplitude-QSVM ---
    norm_tr_raw = np.linalg.norm(X_raw[tr], axis=1, keepdims=True)
    norm_te_raw = np.linalg.norm(X_raw[te], axis=1, keepdims=True)
    X_tr_amp_raw = X_raw[tr] / np.maximum(norm_tr_raw, 1e-12)
    X_te_amp_raw = X_raw[te] / np.maximum(norm_te_raw, 1e-12)
    K_tr_amp_raw = (X_tr_amp_raw @ X_tr_amp_raw.T)**2
    K_te_amp_raw = (X_te_amp_raw @ X_tr_amp_raw.T)**2
    
    svc_amp_raw = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_amp_raw, y_tr)
    p_amp_raw = svc_amp_raw.predict(K_te_amp_raw)
    prob_amp_raw = svc_amp_raw.predict_proba(K_te_amp_raw)[:, 1]
    oof_preds['Raw-Amplitude-QSVM (Collapsed)'][te] = p_amp_raw
    oof_probs['Raw-Amplitude-QSVM (Collapsed)'][te] = prob_amp_raw
    
    # --- 3. Repaired Amplitude-QSVM (Standardized) ---
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_raw[tr])
    X_te_sc = scaler.transform(X_raw[te])
    norm_tr_sc = np.linalg.norm(X_tr_sc, axis=1, keepdims=True)
    norm_te_sc = np.linalg.norm(X_te_sc, axis=1, keepdims=True)
    X_tr_amp_sc = X_tr_sc / np.maximum(norm_tr_sc, 1e-12)
    X_te_amp_sc = X_te_sc / np.maximum(norm_te_sc, 1e-12)
    K_tr_amp_sc = (X_tr_amp_sc @ X_tr_amp_sc.T)**2
    K_te_amp_sc = (X_te_amp_sc @ X_tr_amp_sc.T)**2
    
    svc_amp_sc = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_amp_sc, y_tr)
    p_amp_sc = svc_amp_sc.predict(K_te_amp_sc)
    prob_amp_sc = svc_amp_sc.predict_proba(K_te_amp_sc)[:, 1]
    oof_preds['Repaired-Amplitude-QSVM (Standardized)'][te] = p_amp_sc
    oof_probs['Repaired-Amplitude-QSVM (Standardized)'][te] = prob_amp_sc
    
    # --- 4. Angle-QKNN (k=5) ---
    D_tr_ang = np.sqrt(np.maximum(2.0 * (1.0 - np.sqrt(np.clip(K_tr_ang, 0.0, 1.0))), 0.0))
    D_te_ang = np.sqrt(np.maximum(2.0 * (1.0 - np.sqrt(np.clip(K_te_ang, 0.0, 1.0))), 0.0))
    qknn_ang = KNeighborsClassifier(n_neighbors=5, metric='precomputed').fit(D_tr_ang, y_tr)
    p_qknn_ang = qknn_ang.predict(D_te_ang)
    prob_qknn_ang = qknn_ang.predict_proba(D_te_ang)[:, 1]
    oof_preds['Angle-QKNN (k=5)'][te] = p_qknn_ang
    oof_probs['Angle-QKNN (k=5)'][te] = prob_qknn_ang
    
    # --- 5. Amplitude-QKNN (k=5) ---
    D_tr_amp = np.sqrt(np.maximum(2.0 * (1.0 - np.clip(X_tr_amp_sc @ X_tr_amp_sc.T, -1.0, 1.0)), 0.0))
    D_te_amp = np.sqrt(np.maximum(2.0 * (1.0 - np.clip(X_te_amp_sc @ X_tr_amp_sc.T, -1.0, 1.0)), 0.0))
    qknn_amp = KNeighborsClassifier(n_neighbors=5, metric='precomputed').fit(D_tr_amp, y_tr)
    p_qknn_amp = qknn_amp.predict(D_te_amp)
    prob_qknn_amp = qknn_amp.predict_proba(D_te_amp)[:, 1]
    oof_preds['Amplitude-QKNN (k=5)'][te] = p_qknn_amp
    oof_probs['Amplitude-QKNN (k=5)'][te] = prob_qknn_amp
    
    # --- 6. Classical RBF-SVM ---
    svc_rbf = SVC(kernel='rbf', C=1.0, probability=True, random_state=42).fit(X_tr_sc, y_tr)
    p_rbf = svc_rbf.predict(X_te_sc)
    prob_rbf = svc_rbf.predict_proba(X_te_sc)[:, 1]
    oof_preds['Classical RBF-SVM'][te] = p_rbf
    oof_probs['Classical RBF-SVM'][te] = prob_rbf
    
    # --- 7. Classical KNN (k=5) ---
    knn_cls = KNeighborsClassifier(n_neighbors=5).fit(X_tr_sc, y_tr)
    p_knn = knn_cls.predict(X_te_sc)
    prob_knn = knn_cls.predict_proba(X_te_sc)[:, 1]
    oof_preds['Classical KNN (k=5)'][te] = p_knn
    oof_probs['Classical KNN (k=5)'][te] = prob_knn

print("Completed 5-fold cross validation successfully.")


Running 5-fold CV across all Sodar models and adaptations...


Completed 5-fold cross validation successfully.


In [4]:
# Compute full out-of-fold metrics
records = []
for m in models:
    preds = oof_preds[m]
    probs = oof_probs[m]
    
    acc = accuracy_score(y, preds)
    sens = recall_score(y, preds, zero_division=0)
    spec = recall_score(1 - y, 1 - preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    auc = roc_auc_score(y, probs)
    
    records.append({
        'Model': m,
        'Dataset': 'Pima Indians Diabetes',
        'Accuracy': f"{acc*100:.2f}%",
        'Sensitivity': f"{sens*100:.2f}%",
        'Specificity': f"{spec*100:.2f}%",
        'ROC_AUC': f"{auc:.4f}",
        'F1_Score': f"{f1*100:.2f}%"
    })

df_sodar_metrics = pd.DataFrame(records)
os.makedirs('recheck/results', exist_ok=True)
csv_out = 'recheck/results/paper06_sodar_recheck_metrics.csv'
df_sodar_metrics.to_csv(csv_out, index=False)
print(f"Saved recheck results to {csv_out}")
display(df_sodar_metrics)

with open('recheck/WORKLOG_RECHECK.md', 'a', encoding='utf-8') as f:
    f.write(f'''
## Stage 5.6: Sodar Traffic QML Adaptation (R09)
- Noted: Original ISCX VPN-nonVPN traffic dataset (99.8% claimed) is non-diabetes and marked NOT_TESTABLE_HERE.
- Diagnosed Amplitude-QSVM collapse: raw positive orthant vectors cause rank-1 spectral dominance ({top1_raw:.1f}% top eigenvalue) and near-constant Gram matrix ({np.mean(offdiag_raw):.4f} mean off-diagonal), shrinking dual coefficients to zero and collapsing to majority class (0.0% sensitivity, 100% specificity).
- Repaired Amplitude-QSVM via in-fold standardization: condition number improved from {cond_raw:.1e} to {cond_sc:.1e}, restoring sensitivity to {float(df_sodar_metrics.loc[df_sodar_metrics['Model']=='Repaired-Amplitude-QSVM (Standardized)', 'Sensitivity'].values[0].replace('%','')):.1f}%.
- Angle-QSVM achieved {df_sodar_metrics.loc[df_sodar_metrics['Model']=='Angle-QSVM', 'Accuracy'].values[0]} accuracy and {df_sodar_metrics.loc[df_sodar_metrics['Model']=='Angle-QSVM', 'ROC_AUC'].values[0]} ROC-AUC, competitive with Classical RBF-SVM ({df_sodar_metrics.loc[df_sodar_metrics['Model']=='Classical RBF-SVM', 'Accuracy'].values[0]}).
''')
print("Appended Stage 5.6 to recheck/WORKLOG_RECHECK.md")


Saved recheck results to recheck/results/paper06_sodar_recheck_metrics.csv


,Model,Dataset,Accuracy,Sensitivity,Specificity,ROC_AUC,F1_Score
0,Angle-QSVM,Pima Indians Diabetes,75.52%,54.48%,86.80%,0.8255,60.83%
1,Raw-Amplitude-QSVM (Collapsed),Pima Indians Diabetes,65.10%,0.00%,100.00%,0.6755,0.00%
2,Repaired-Amplitude-QSVM (Standardized),Pima Indians Diabetes,64.45%,14.18%,91.40%,0.6496,21.78%
3,Angle-QKNN (k=5),Pima Indians Diabetes,72.01%,53.36%,82.00%,0.7664,57.09%
4,Amplitude-QKNN (k=5),Pima Indians Diabetes,74.61%,61.57%,81.60%,0.7772,62.86%
5,Classical RBF-SVM,Pima Indians Diabetes,76.17%,56.34%,86.80%,0.8147,62.27%
6,Classical KNN (k=5),Pima Indians Diabetes,72.79%,55.22%,82.20%,0.7769,58.61%


Appended Stage 5.6 to recheck/WORKLOG_RECHECK.md
